# Fine-tuning Llama 3.1 8B Instruct dengan PEFT QLoRA (Klasifikasi Pantun)

Notebook ini mengikuti alur eksperimen pada indobert.ipynb dengan penyesuaian untuk model kausal LLM menggunakan QLoRA 4-bit.

Catatan penting:
- Model default: meta-llama/Llama-3.1-8B-Instruct (butuh akses Hugging Face).
- Direkomendasikan GPU Colab T4/L4/A100 dengan runtime CUDA.
- Output dipisah per skenario: full vs isi_only.

In [ ]:
# Instal dependensi (jalankan sekali).
!pip -q install -U \
  "transformers>=4.44,<5" \
  "datasets>=2.19,<4" \
  "evaluate>=0.4,<0.5" \
  "accelerate>=0.33,<2" \
  "scikit-learn>=1.3,<2" \
  "seaborn>=0.13,<0.14" \
  "peft>=0.12,<1" \
  "bitsandbytes>=0.43,<0.47"

In [ ]:
import os
import re
import json
import time
import random
import zipfile
import inspect
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Konfigurasi path dan skenario.
IN_COLAB = False
try:
    from google.colab import drive, files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SCENARIO = 'full'  # ganti ke 'isi_only' untuk ablation
if SCENARIO not in {'full', 'isi_only'}:
    raise ValueError("SCENARIO harus 'full' atau 'isi_only'")

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Penelitian_Pantun')
    DATA_PATH = Path('/content/drive/MyDrive/Penelitian_Pantun/Dataset_Final_Klasifikasi.csv')
    if not DATA_PATH.exists():
        DATA_PATH = Path('/content/Dataset_Final_Klasifikasi.csv')
else:
    BASE_DIR = Path('./outputs/article3/llama8b_qlora')
    DATA_PATH = Path('../../data/article3/Dataset_Final_Klasifikasi.csv')

RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = BASE_DIR / f'run_{SCENARIO}_{RUN_TS}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('SCENARIO:', SCENARIO)
print('DATA_PATH:', DATA_PATH)
print('RUN_DIR:', RUN_DIR)

In [ ]:
# Load dataset dan bangun text_input.
df = pd.read_csv(DATA_PATH, sep=';', engine='python', on_bad_lines='skip')
required_cols = ['text_pantun', 'Kategori Pakar']
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f'Kolom wajib tidak ditemukan: {c}')

df = df[required_cols].copy()
df['text_pantun'] = df['text_pantun'].astype(str)
df['Kategori Pakar'] = df['Kategori Pakar'].astype(str).str.strip()

def build_input_text(text, scenario='full'):
    text = re.sub(r'\\n|\r\n|\r', '\n', text)
    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    if scenario == 'isi_only':
        selected = lines[-2:] if len(lines) >= 2 else lines
    else:
        selected = lines
    normalized = ' '.join(selected)
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

df['text_input'] = df['text_pantun'].apply(lambda x: build_input_text(x, SCENARIO))
df = df[(df['text_input'] != '') & (df['Kategori Pakar'] != '')]
df = df.reset_index(drop=True)

print('Jumlah data setelah cleaning:', len(df))
print('Skenario input:', SCENARIO)
print('Contoh text_input:', df['text_input'].iloc[0][:200])
print('Distribusi label:')
print(df['Kategori Pakar'].value_counts())

In [ ]:
# Label encoding dan split stratified.
label_list = sorted(df['Kategori Pakar'].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

df['label'] = df['Kategori Pakar'].map(label2id)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df['label']
)

split_meta = {
    'seed': SEED,
    'scenario': SCENARIO,
    'input_column': 'text_input',
    'n_total': int(len(df)),
    'n_train': int(len(train_df)),
    'n_valid': int(len(valid_df)),
    'n_test': int(len(test_df)),
    'labels': label_list,
}
(RUN_DIR / 'split_metadata.json').write_text(json.dumps(split_meta, indent=2, ensure_ascii=False), encoding='utf-8')

print('Train:', train_df.shape, 'Valid:', valid_df.shape, 'Test:', test_df.shape)

In [ ]:
# Tokenizer dan dataset HF.
MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_ds = Dataset.from_pandas(train_df[['text_input', 'label']], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df[['text_input', 'label']], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['text_input', 'label']], preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(batch['text_input'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_ds)

In [ ]:
# Class weights + weighted loss trainer.
class_counts = train_df['label'].value_counts().sort_index()
num_classes = len(label_list)
num_train = len(train_df)

class_weights = torch.tensor(
    [num_train / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Load Llama 8B 4-bit + pasang QLoRA.
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Metrics dan training arguments (kompatibel antar versi transformers).
metric_accuracy = evaluate.load('accuracy')
metric_f1 = evaluate.load('f1')
metric_precision = evaluate.load('precision')
metric_recall = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_accuracy.compute(predictions=preds, references=labels)['accuracy']
    f1_macro = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    f1_weighted = metric_f1.compute(predictions=preds, references=labels, average='weighted')['f1']
    precision_macro = metric_precision.compute(predictions=preds, references=labels, average='macro')['precision']
    recall_macro = metric_recall.compute(predictions=preds, references=labels, average='macro')['recall']
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
    }

OUTPUT_DIR = RUN_DIR / 'adapter'
SAVE_STEPS = 100
EVAL_STEPS = 100

ta_params = inspect.signature(TrainingArguments.__init__).parameters
train_args_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    logging_strategy='steps',
    logging_steps=25,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    seed=SEED,
)
if 'eval_strategy' in ta_params:
    train_args_kwargs['eval_strategy'] = 'steps'
else:
    train_args_kwargs['evaluation_strategy'] = 'steps'
training_args = TrainingArguments(**train_args_kwargs)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
# Train.
start_time = time.perf_counter()
train_result = trainer.train()
end_time = time.perf_counter()

training_time_seconds = end_time - start_time
training_time_minutes = training_time_seconds / 60

training_time_data = {
    'training_time_seconds': training_time_seconds,
    'training_time_minutes': training_time_minutes,
    'train_samples': len(train_df),
    'valid_samples': len(valid_df),
    'test_samples': len(test_df),
    'timestamp': RUN_TS,
    'scenario': SCENARIO,
}
(RUN_DIR / 'training_time.json').write_text(json.dumps(training_time_data, indent=2), encoding='utf-8')
print(f'Training selesai dalam {training_time_seconds:.2f} detik ({training_time_minutes:.2f} menit).')

In [ ]:
# Evaluasi test + laporan.
test_output = trainer.predict(test_ds)
y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

results = {
    'accuracy': float(acc),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'precision_macro': float(precision_macro),
    'recall_macro': float(recall_macro),
    'training_time_seconds': float(training_time_seconds),
    'training_time_minutes': float(training_time_minutes),
}

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(num_classes)],
    output_dict=True,
    zero_division=0,
)

(RUN_DIR / 'test_metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
(RUN_DIR / 'classification_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
pd.DataFrame([results]).to_csv(RUN_DIR / 'test_metrics.csv', index=False)

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=[id2label[i] for i in range(num_classes)], columns=[id2label[i] for i in range(num_classes)])
cm_df.to_csv(RUN_DIR / 'confusion_matrix.csv', index=True)

fig_dir = RUN_DIR / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
cm_png_path = fig_dir / 'confusion_matrix.png'
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Llama 8B QLoRA (Test Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(cm_png_path, dpi=200, bbox_inches='tight')
plt.show()

print('Test Accuracy:', round(acc, 4))
print('Test Macro F1:', round(f1_macro, 4))
print('Confusion matrix PNG:', cm_png_path)

In [ ]:
# Simpan adapter QLoRA + tokenizer + config run.
adapter_dir = RUN_DIR / 'best_adapter'
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

config_dump = {
    'scenario': SCENARIO,
    'input_column': 'text_input',
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'num_classes': num_classes,
    'class_weights': [float(x) for x in class_weights.tolist()],
    'qlora': {
        'r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        'load_in_4bit': True,
        'quant_type': 'nf4',
    },
    'training_args': {
        'learning_rate': training_args.learning_rate,
        'per_device_train_batch_size': training_args.per_device_train_batch_size,
        'per_device_eval_batch_size': training_args.per_device_eval_batch_size,
        'gradient_accumulation_steps': training_args.gradient_accumulation_steps,
        'num_train_epochs': training_args.num_train_epochs,
        'weight_decay': training_args.weight_decay,
    },
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(config_dump, indent=2), encoding='utf-8')
print('Adapter tersimpan di:', adapter_dir)

In [ ]:
# Kompres report-only (tanpa adapter/model besar).
zip_path = BASE_DIR / f'llama8b_qlora_{SCENARIO}_report_only_{RUN_TS}.zip'
report_files = [
    RUN_DIR / 'test_metrics.json',
    RUN_DIR / 'test_metrics.csv',
    RUN_DIR / 'classification_report.json',
    RUN_DIR / 'confusion_matrix.csv',
    RUN_DIR / 'training_time.json',
    RUN_DIR / 'split_metadata.json',
    RUN_DIR / 'run_config.json',
]
figure_files = sorted((RUN_DIR / 'figures').glob('*.png')) if (RUN_DIR / 'figures').exists() else []
included_files = [p for p in report_files + figure_files if p.exists()]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for full_path in included_files:
        zf.write(full_path, full_path.relative_to(RUN_DIR))

print('ZIP tersimpan:', zip_path)
print('Jumlah file dalam ZIP:', len(included_files))
if IN_COLAB:
    print('Jalankan cell berikutnya untuk download ZIP.')

In [ ]:
# Opsional download dari Colab.
if IN_COLAB:
    files.download(str(zip_path))
else:
    print('Bukan environment Colab. ZIP ada di:', zip_path)